# Clase 4 — De HTTP a la primera aplicación FastAPI

> **Pregunta guía:** ¿cómo puede otro programa pedirle datos a una función Python sin
importar directamente nuestro archivo?

En la clase 3 hicimos reproducible el ambiente. Hoy pondremos una aplicación dentro de
ese tipo de proyecto y conversaremos con ella mediante HTTP.

**Resultado observable:** podrás explicar qué problema resuelve una API, descomponer un
request y un response HTTP, decidir dónde viaja cada dato de entrada y levantar un servidor
local con dos endpoints `GET` que produzcan respuestas `200`, `404` y `422`.


## Antes de comenzar

Actualiza el repositorio público desde su raíz:

```bash
git status
git switch main
git pull
```

La práctica se realiza en una copia local no calificable. No requiere rama, commit,
push, PR ni entrega en Canvas.

**Prerrequisitos:** reconocer un ambiente virtual y los archivos `pyproject.toml` y
`uv.lock`; ejecutar `uv sync --locked` y `uv run`.


## 1. ¿Qué es una API?

Una **Interfaz de Programa de Aplicación** (API):

- Define las reglas que se deben seguir para comunicarse con otros sistemas de software.
- Es un conjunto de definiciones y protocolos que permiten que diferentes aplicaciones se
  comuniquen entre sí. En términos simples, es un intermediario que permite que dos
  aplicaciones hablen entre sí.

Los desarrolladores exponen o crean una API para que otras aplicaciones puedan comunicarse
con sus aplicaciones mediante programación. Por ejemplo, una aplicación de planilla de
horarios puede recibir el nombre completo de un empleado y un rango de fechas, procesar
internamente la planilla y devolver la cantidad de horas trabajadas.

Se puede pensar en una API web como una puerta de enlace entre los clientes y los recursos
de la Web.

### Contrato de una API

El **contrato** es el acuerdo que describe qué se puede pedir, cómo se pide y qué respuesta
se recibe cuando funciona o cuando algo sale mal. Como un menú, permite que el cliente use
el servicio sin conocer cómo está construido por dentro.

**Ejemplo de contrato:** `GET /viajes/42` significa “consulta el viaje con identificador
42”; si existe, devuelve sus datos en JSON; si no existe, devuelve un código `404`.

### Clientes

- Usuarios o sistemas de software que desean acceder a información desde la Web.
- Un navegador, una aplicación móvil, un dashboard u otro programa pueden actuar como
  cliente. Por ejemplo, una app de transporte consulta la duración de un viaje para
  mostrársela a la persona que la está usando.

### Recursos

- Información que diferentes aplicaciones proporcionan a sus clientes.
- Pueden ser imágenes, videos, texto, números o cualquier tipo de datos.
- La máquina encargada de entregar el recurso al cliente también recibe el nombre de
  servidor.
- Las organizaciones utilizan las API para compartir recursos y proporcionar servicios web,
  manteniendo controlado quién puede acceder a ellos.

**Ejemplos de recursos:** un viaje (`/viajes/42`), una colección de viajes
(`/viajes/`), un usuario (`/usuarios/7`) o el resultado de una predicción
(`/predicciones/abc123`).

En esta clase construiremos una API web: una API cuyos mensajes viajan mediante HTTP.


![Diagrama de una API como intermediario entre cliente y servidor](../assets/modulo-01-fundamentos/clase-04/api.png)

*Figura 1. ¿Qué es una API?*



# Arquitecturas de APIs

Hasta ahora usamos API como nombre general para una interfaz entre programas. Existen
distintas maneras de diseñar esa comunicación. En esta clase nos interesa REST porque
trabaja naturalmente con HTTP, recursos y operaciones conocidas.

## 2.1 REST (Representational State Transfer)

REST es un estilo de arquitectura: un conjunto de decisiones para organizar una API, no
una biblioteca que instalemos. Una API REST identifica recursos y usa métodos HTTP para
consultarlos o modificarlos. Cada solicitud debe llevar la información necesaria para que
el servidor la entienda.

### Principios que usaremos

- **Estado sin sesión:** el servidor procesa cada solicitud con la información que recibe;
  no depende de recordar una solicitud anterior.
- **Interfaz uniforme:** rutas y métodos siguen convenciones predecibles.
- **Recursos identificables:** cada recurso tiene una dirección o URL que permite pedirlo.

## 2.2 Otras arquitecturas

REST no es la única forma de diseñar una API. Estas alternativas aparecen con frecuencia:

- **SOAP** (*Simple Object Access Protocol*) es un protocolo con mensajes estructurados,
  tradicionalmente escritos en XML. Define reglas formales y se usa en muchos sistemas
  empresariales.
- **GraphQL** permite que el cliente describa exactamente qué campos necesita. En lugar de
  varias rutas REST, suele trabajar con un punto de entrada que responde esa consulta.
- **gRPC** (*Google Remote Procedure Call*) permite que un programa invoque una operación
  en otro programa con comunicación binaria eficiente. Es común entre microservicios.

Las mencionamos para reconocerlas; hoy implementaremos una API web con convenciones REST.

# ¿Cómo funcionan las API RESTful?

La función básica es parecida a navegar por Internet: cuando el cliente requiere un
recurso, se pone en contacto con el servidor mediante la API. El cliente consulta la
documentación, construye la solicitud, el servidor la procesa y devuelve una respuesta.
La respuesta indica si la solicitud tuvo éxito y puede incluir el recurso pedido.

![Diagrama conceptual de una API REST con cliente, servidor, recursos y métodos HTTP](https://www.astera.com/wp-content/uploads/2020/01/rest.png)

*Figura 2. Elementos habituales de una API REST. Fuente: [Astera](https://www.astera.com/wp-content/uploads/2020/01/rest.png).*

Esta secuencia prepara la siguiente pregunta: ¿qué contiene exactamente una solicitud?


## 3. ¿Qué contiene el request del cliente?

Un request es el mensaje con el que el cliente pide algo. Contiene una línea inicial,
headers, una línea vacía y un body opcional. Observa primero sus piezas:

La línea inicial combina **método**, **destino** y versión de HTTP:

```text
GET /viajes/42?pasajeros=2 HTTP/1.1
Host: 127.0.0.1:8000
Accept: application/json
```

### 3.1 URL: dónde está el recurso u operación

La URL completa será `http://127.0.0.1:8000/viajes/42?pasajeros=2`.

| Parte | Función | Ejemplo |
|---|---|---|
| Esquema | Protocolo usado para comunicarse. | `http` |
| Host | Máquina o nombre donde está el servicio. | `127.0.0.1` |
| Puerto | Programa de red dentro del host. | `8000` |
| Path | Ubicación lógica del recurso u operación. | `/viajes/42` |
| Query string | Opciones escritas después de `?`. | `pasajeros=2` |

Un **endpoint** combina método HTTP y path, por ejemplo `GET /viajes/{viaje_id}`. La
URL concreta usa ese endpoint con el identificador `42`; endpoint, path y URL completa
no son sinónimos.


### 3.2 Método: qué intención tiene el request

El **método HTTP** expresa la intención de una solicitud. Cuando una API administra
recursos que pueden conservarse, las operaciones se suelen resumir como **CRUD**:
crear, consultar, actualizar y eliminar. CRUD es un mapa conceptual, no un protocolo;
una API de predicción también puede ofrecer operaciones que no son CRUD.

| Intención | Método habitual | ¿Cambia estado? | Ejemplo |
|---|---|:---:|---|
| Consultar | `GET` | No | `GET /viajes/42` |
| Crear o solicitar procesamiento | `POST` | Usualmente sí | `POST /predicciones` |
| Reemplazar por completo | `PUT` | Sí | `PUT /viajes/42` |
| Modificar una parte | `PATCH` | Sí | `PATCH /viajes/42` |
| Eliminar | `DELETE` | Sí | `DELETE /viajes/42` |

Una operación **idempotente** puede repetirse y dejar el mismo estado final que una sola
ejecución. `GET`, `PUT` y `DELETE` se diseñan con esa propiedad; `POST` no la garantiza.
Hoy sólo implementaremos `GET`. En la clase 5 construiremos un `POST` con body JSON.

### 3.3 Headers, parámetros y body: dónde viajan los datos

Los **headers** describen el mensaje. Un **parámetro** completa o ajusta la operación;
el **body** transporta datos estructurados cuando no caben naturalmente en la URL.

| Entrada | Dónde aparece | Cuándo conviene | Ejemplo |
|---|---|---|---|
| **Path parameter** | Dentro de la ruta. | Identifica un recurso específico. | `42` en `/viajes/42` |
| **Query parameter** | Después de `?`. | Filtra, pagina o agrega opciones. | `?pasajeros=2&detalle=true` |
| **Header** | En los metadatos. | Describe formato, credenciales o contexto. | `Accept: application/json` |
| **Body** | Después de headers y línea vacía. | Envía una estructura completa. | JSON con datos de un viaje |

La ruta responde **qué recurso**; el query, **con qué opciones**; el body, **qué datos
estructurados**. No coloques información sensible en una URL: puede aparecer en logs e
historiales. Hoy usaremos path y query; Pydantic validará el body en la clase 5.


## 4. ¿Qué contiene el response del servidor?

```text
HTTP/1.1 200 OK
content-type: application/json

{"viaje_id":42,"pasajeros":2}
```

La primera línea incluye versión, código numérico y frase descriptiva. Después aparecen
headers, una línea vacía y el body. El response no repite la operación: comunica qué
ocurrió al procesarla.

| Parte | Qué comunica |
|---|---|
| Código de estado | Resultado general de la operación. |
| Headers | Metadatos, por ejemplo el tipo de contenido. |
| Body | Datos devueltos; hoy será JSON. |

### 4.1 Línea de estado y familias de códigos

El primer dígito ubica el resultado en una familia. La familia orienta el diagnóstico;
el código específico precisa qué ocurrió:

![Cinco familias de códigos HTTP: 1xx información, 2xx éxito, 3xx redirección, 4xx solicitud no resuelta y 5xx falla del servidor](../assets/modulo-01-fundamentos/clase-04/familias-codigos-http.svg)

*Figura 3. Familias de códigos de estado HTTP.*

Hoy observaremos `200`, `404` y `422`; los demás preparan contratos posteriores:

| Código | Significado en la práctica | Ejemplo |
|---:|---|---|
| `200 OK` | La solicitud fue válida. | Existe el endpoint y los parámetros son correctos. |
| `201 Created` | Se creó un recurso o resultado nuevo. | Respuesta habitual de un `POST` de creación. |
| `400 Bad Request` | La solicitud no tiene una forma que el servidor puede procesar. | JSON mal formado. |
| `401 Unauthorized` | Falta una identidad verificable. | Falta autenticación para un recurso protegido. |
| `403 Forbidden` | La identidad existe, pero no tiene permiso. | Un usuario no puede realizar esa operación. |
| `404 Not Found` | No existe una ruta que coincida. | Pedir `/no-existe`. |
| `422 Unprocessable Content` | La ruta existe, pero un dato no cumple el contrato. | Usar texto donde `viaje_id` debe ser entero. |
| `500 Internal Server Error` | El servidor falló al procesar una solicitud válida. | Error no controlado en el programa. |

Un código no cuenta toda la historia: también se inspeccionan headers y body.


### 4.2 Headers y body de la respuesta

Los headers indican cómo interpretar el mensaje; el body contiene la representación del
recurso o el detalle del error. **JSON** es un formato de texto interoperable. FastAPI
puede convertir automáticamente
un diccionario retornado por Python en un cuerpo JSON.

| Python | JSON |
|---|---|
| `{'activo': True}` | `{"activo": true}` |
| Permite tuplas y otros objetos de Python. | Sólo admite tipos definidos por JSON. |
| Existe en memoria del proceso. | Viaja como texto/bytes entre programas. |

FastAPI también agrega el header `content-type: application/json`, para que el cliente
sepa cómo interpretar el cuerpo.


In [1]:
import json

respuesta_python = {"viaje_id": 42, "pasajeros": 2, "activo": True}
cuerpo_json = json.dumps(respuesta_python)
print(cuerpo_json)
print(type(respuesta_python).__name__, "→", type(cuerpo_json).__name__)


{"viaje_id": 42, "pasajeros": 2, "activo": true}
dict → str


## 5. ¿Qué son los métodos de autenticación de la API RESTful?

- Un servicio web RESTful debe autenticar las solicitudes antes de poder enviar una respuesta.
- La autenticación es el proceso de verificar una identidad.
- Los clientes de los servicios RESTful deben demostrar su identidad al servidor para establecer confianza.

La API RESTful tiene tres métodos comunes de autenticación:

### 1. Autenticación HTTP

HTTP define algunos esquemas de autenticación que se pueden utilizar directamente cuando se implementa la API REST. A continuación, se indican dos de estos esquemas:

#### 1.1 Autenticación básica

En la autenticación básica, el cliente envía el nombre y la contraseña del usuario en el encabezado de la solicitud. Los codifica con base64, que es una técnica de codificación que convierte el par en un conjunto de 64 caracteres para su transmisión segura.

### 2. Claves de la API

- En este enfoque, el servidor asigna un valor único generado a un cliente por primera vez.
- Cada vez que el cliente intenta acceder a los recursos, utiliza la clave de API única para su verificación.
- Las claves de API son menos seguras debido a que el cliente debe transmitir la clave, lo que la vuelve vulnerable al robo de red.

### 3. OAuth

- **OAuth** combina contraseñas y tokens para el acceso de inicio de sesión de alta seguridad a cualquier sistema.
- El servidor primero solicita una contraseña y luego solicita un token adicional para completar el proceso de autorización.
- Puede verificar el token en cualquier momento y, también, a lo largo del tiempo, con un alcance y duración específicos.
En este curso usaremos autenticación mediante **token**. En esta clase sólo reconocemos
la idea; la implementaremos más adelante, cuando el servicio de predicciones tenga un
endpoint que necesite protección.


## Nombramiento de endpoints y relación con recursos

En REST, un **endpoint** representa una operación sobre un recurso. Los nombres deben
ser claros, intuitivos y consistentes:

- Usa sustantivos en plural para colecciones: `/viajes/`.
- Evita verbos en el nombre; la acción se expresa con `GET`, `POST`, `PUT` o `DELETE`.
- Mantén un estilo consistente y considera versionar la API: `/api/v1/viajes/`.

Ejemplos:

```text
GET    /api/v1/viajes/       # lista viajes
POST   /api/v1/viajes/       # crea un viaje
GET    /api/v1/viajes/42     # consulta un viaje
DELETE /api/v1/viajes/42     # elimina un viaje
GET    /api/v1/usuarios/7/viajes/  # viajes de un usuario
```

## 🔑 Conceptos Clave: Path, Query y Body Parameters

Una **URL** puede componerse de:

- **Ruta (path):** `/items/42`
- **Consulta (query string):** `?filter=active`
- **Combinados:** `/items/42?filter=active`

### 1. 📍 Path Parameters

Se usan para identificar recursos específicos dentro de la URL.

Un **decorador de Python** es una función que recibe otra función y devuelve una función
modificada o registrada. La sintaxis `@decorador` aplica ese comportamiento a la función
que aparece inmediatamente debajo. FastAPI usa esta herramienta para asociar una función
con una ruta y un método HTTP.

En FastAPI se definen entre `{}` en el decorador.

```python
@app.get("/items/{item_id}")
async def read_item(item_id: int):
    return {"item_id": item_id}
```

🔹 **Cuándo usarlos:** Cuando el valor es parte de la estructura del recurso (ej. un ID).

👉 URL de ejemplo: `/items/42`

### 2. 🔎 Query Parameters

Se envían después del `?` en la URL. Son opcionales y se usan para filtrar o ajustar resultados.

```python
@app.get("/items/")
async def read_items(skip: int = 0, limit: int = 10):
    return {"skip": skip, "limit": limit}
```

🔹 **Cuándo usarlos:** Para filtros, búsqueda, paginación o configuraciones opcionales.

👉 URL de ejemplo: `/items/?skip=0&limit=10`

### 3. 📦 Request Body

Se usan para enviar datos más complejos (ej. JSON) en el cuerpo de la petición.

En FastAPI se definen con modelos de **Pydantic**.

```python
from pydantic import BaseModel

class Item(BaseModel):
    name: str
    description: str | None = None
    price: float
    tax: float | None = None

@app.post("/items/")
async def create_item(item: Item):
    return item
```

🔹 **Cuándo usarlos:** Para enviar datos estructurados al crear o actualizar recursos.

👉 Ejemplo: crear un `item` enviando un JSON como:

```json
{
  "name": "Camiseta",
  "price": 19.99,
  "description": "Camiseta 100% algodón"
}
```

## Type hints y clases como tipos

Un **type hint** es una anotación que indica el tipo esperado de una variable, parámetro o
resultado. Python sigue siendo dinámico: la anotación documenta y ayuda a las herramientas,
pero FastAPI también puede usarla para convertir y validar entradas.

```python
def estimar(viaje_id: int, distancia_km: float) -> float:
    return distancia_km * 2.5
```

Una **clase como tipo** agrupa varios datos relacionados en una sola estructura. Esta idea
prepara los modelos Pydantic de la próxima clase:

```python
class Viaje:
    origen: str
    destino: str
    pasajeros: int
```

**Lectura obligatoria para la próxima clase:** [Clases como tipos y modelos Pydantic](https://fastapi.tiangolo.com/python-types/#pydantic-models).

## Explorar APIs públicas con Postman

Vamos a explorar algunas API públicas y ver todos los componentes que acabamos de estudiar.

[Opción 1: Public APIs](https://github.com/public-apis/public-apis)

[Opción 2: Public API Lists](https://github.com/public-api-lists/public-api-lists)

Ahora vamos a [descargar Postman](https://www.postman.com/downloads/) y explorar las APIs
con esta herramienta.

[Rick and Morty API](https://rickandmortyapi.com/)

[Dad Jokes API](https://icanhazdadjoke.com/api#search-for-dad-jokes)

1. Abre Postman y crea una pestaña de request.
2. Usa `GET https://rickandmortyapi.com/api/character/1` y después cambia el id a `2`.
3. Usa `GET https://icanhazdadjoke.com/` y agrega `Accept: application/json` en headers.
4. Pulsa **Send** y describe URL, headers, status y body en cada respuesta.
5. Compara lo observado con la estructura de request y response que acabamos de estudiar.

Después de esta exploración pasaremos a FastAPI para construir un servicio propio.


## 6. Primera aplicación con FastAPI

### 6.1 FastAPI, Uvicorn y ASGI no son lo mismo

Un **framework** proporciona la estructura dentro de la que escribimos una aplicación y
decide cuándo llamar nuestro código. Un **type hint** es una anotación que comunica el
tipo esperado, por ejemplo `viaje_id: int`. FastAPI conecta ambas ideas con HTTP.

| Pieza | Responsabilidad | No hace por sí sola |
|---|---|---|
| **FastAPI** | Framework para declarar operaciones mediante funciones, decoradores y type hints; valida entradas, convierte salidas y genera OpenAPI. | No abre por sí solo el puerto que recibe tráfico HTTP. |
| **Uvicorn** | Servidor ASGI: escucha en una dirección y puerto, traduce la conexión HTTP a eventos para la aplicación y devuelve sus eventos como response. | No decide las rutas ni la lógica de negocio. |
| **ASGI** | Estándar de comunicación entre el servidor y la aplicación Python. | No es un servidor ni un framework que instalemos para escribir endpoints. |
| **OpenAPI** | Descripción estructurada de rutas, parámetros, cuerpos y respuestas. | No ejecuta la aplicación; FastAPI la usa para generar `/docs`. |

ASGI significa *Asynchronous Server Gateway Interface*. Es una frontera común: Uvicorn
recibe HTTP y entrega a la aplicación un contexto de conexión y eventos; la aplicación
recibe esos eventos y envía eventos de respuesta. Gracias a ese acuerdo, FastAPI no
necesita implementar directamente conexiones de red, puertos o el protocolo HTTP.

Sigue el recorrido de una solicitud local:

![Flujo desde un cliente hacia Uvicorn, FastAPI y una función Python, con una respuesta HTTP y JSON de regreso](../assets/modulo-01-fundamentos/clase-04/flujo-http-fastapi.svg)

*Figura 4. Implementación local del contrato con Uvicorn y FastAPI.*

1. Uvicorn escucha en `127.0.0.1:8000`.
2. Recibe el request HTTP y lo representa conforme a ASGI.
3. FastAPI compara método y path para elegir una operación.
4. FastAPI convierte y valida parámetros usando los type hints.
5. La función Python produce un resultado.
6. FastAPI y Uvicorn lo convierten en un response HTTP para el cliente.

### 6.2 ¿Qué se instala?

La ruta recomendada por la documentación de FastAPI para un proyecto nuevo es instalar
el extra `standard`, que reúne la CLI, Uvicorn y otras herramientas habituales:

```bash
uv add "fastapi[standard]"
```

Los **extras** entre corchetes son dependencias opcionales agrupadas por el paquete; no
son otra edición de FastAPI. El extra `standard` incluye el comando `fastapi` y
`uvicorn[standard]`, por eso permite iniciar el servidor con `uv run fastapi dev`. El
starter ya declara `fastapi[standard]` en
`pyproject.toml` y `uv.lock`, así que hoy **no ejecutes `uv add`**: reconstruye exactamente
lo declarado con `uv sync --locked`.

Fuentes: [entornos y `fastapi[standard]`](https://fastapi.tiangolo.com/virtual-environments/),
[servidores ASGI y Uvicorn](https://fastapi.tiangolo.com/deployment/manually/) y
[especificación ASGI](https://asgi.readthedocs.io/en/latest/specs/main.html).

### 6.3 Iniciar el servidor

Como instalamos el extra estándar, podemos usar el comando corto:

```bash
uv run fastapi dev
```

| Fragmento | Significado |
|---|---|
| `uv run` | Ejecuta con las dependencias del ambiente del proyecto. |
| `fastapi` | Invoca la CLI incluida por `fastapi[standard]`. |
| `dev` | Inicia el servidor en modo desarrollo y descubre `main.py`. |

Usaremos el comando recomendado:

```bash
uv run fastapi dev
```

`fastapi dev` descubre la aplicación y activa el servidor de desarrollo.


### 6.4 Aplicación, decorador y endpoint

```python
from fastapi import FastAPI

app = FastAPI(title="API de viajes del curso")

@app.get("/")
def inicio() -> dict[str, str]:
    return {"mensaje": "API activa"}
```

- `app` representa la aplicación FastAPI.
- `@app.get("/")` es un **decorador**: registra la función siguiente para el método
  `GET` y el path `/`.
- `-> dict[str, str]` es un **type hint**: comunica que el resultado esperado es un
  diccionario cuyas llaves y valores son texto.
- El endpoint es `GET /`; no es solamente `/` ni solamente `inicio()`.

El decorador registra la función al importar el archivo; no necesitamos llamarla a mano
para atender HTTP.


### 6.5 Del mensaje HTTP a parámetros de Python

```python
@app.get("/viajes/{viaje_id}")
def consultar_viaje(viaje_id: int, pasajeros: int = 1) -> dict[str, int]:
    ...
```

| Parámetro | Cómo se reconoce | Ejemplo | Obligatorio |
|---|---|---|:---:|
| Path | Aparece entre llaves en la ruta. | `viaje_id` en `/viajes/42` | Sí |
| Query | Está en la función, pero no en la ruta. | `?pasajeros=2` | No, tiene valor `1` |

El type hint `int` forma parte del contrato: FastAPI convierte texto válido a entero y
responde `422` cuando no puede hacerlo. Esa validación ocurre antes de ejecutar nuestra
función.


## 8. Preparar la práctica local con el ambiente bloqueado

Desde la raíz del repositorio público, copia sólo el script del starter a la zona ignorada
local:

```bash
mkdir -p labs/trabajo-local
mkdir -p labs/trabajo-local/clase-04
cp labs/starters/clase-04-fastapi/main.py labs/trabajo-local/clase-04/main.py
cd labs/trabajo-local/clase-04
```


```text
clase-04/
└── main.py
```

**Checkpoint:** `uv run python --version` debe indicar Python 3.12 y
`uv run fastapi --help` debe terminar sin error.


## 9. Completar el contrato mínimo

Abre `main.py`. El primer endpoint ya muestra la estructura. Completa los `TODO` para
obtener este comportamiento:

| Solicitud | Respuesta esperada |
|---|---|
| `GET /` | `{"mensaje": "API activa"}` |
| `GET /viajes/42` | `{"viaje_id": 42, "pasajeros": 1}` |
| `GET /viajes/42?pasajeros=2` | `{"viaje_id": 42, "pasajeros": 2}` |

Para el segundo endpoint necesitas:

1. registrar `@app.get("/viajes/{viaje_id}")`;
2. definir `viaje_id: int` y `pasajeros: int = 1`;
3. retornar un diccionario con ambos valores.

Guarda el archivo antes de iniciar el servidor.


## 10. Servidor y cliente en dos terminales

**Terminal 1 — servidor** (déjala abierta):

```bash
uv run fastapi dev
```

Debes observar que Uvicorn escucha en `http://127.0.0.1:8000`. Los logs de esta terminal
muestran cada solicitud. `Ctrl+C` detiene el proceso al terminar.

**Terminal 2 — cliente** (abre otra Git Bash/terminal en la misma carpeta):

```bash
curl --version
curl -i http://127.0.0.1:8000/
curl -i "http://127.0.0.1:8000/viajes/42?pasajeros=2"
```

`curl` es un cliente de terminal. La opción `-i` incluye los headers de la respuesta,
para que puedas observar el código, el tipo de contenido y el body; no envía el servidor
a internet. `curl` es una herramienta del sistema, no una dependencia de FastAPI.
Comprueba `curl --version` antes de usarlo. Si tu equipo no lo tiene, abre las mismas
URLs en el navegador o usa la interfaz `/docs`; la idea es observar el request y response,
no aprender a instalar una herramienta adicional.


### 10.1 Provocar respuestas distintas

Ejecuta una solicitud por vez y explica el resultado antes de continuar:

```bash
curl -i http://127.0.0.1:8000/no-existe
curl -i http://127.0.0.1:8000/viajes/no-es-entero
```

- `/no-existe` no coincide con ningún endpoint: `404`.
- `/viajes/no-es-entero` sí coincide con el patrón, pero el valor no cumple `int`: `422`.

**Checkpoint:** relaciona cada salida de la terminal cliente con una línea del log del
servidor. Si no hay log, la solicitud probablemente no llegó al proceso correcto.


## 11. `/docs` y el contrato OpenAPI

Abre en el navegador:

- `http://127.0.0.1:8000/docs`: interfaz para explorar y probar los endpoints;
- `http://127.0.0.1:8000/openapi.json`: descripción estructurada del contrato.

Busca `GET /viajes/{viaje_id}` y responde:

1. ¿Cuál parámetro pertenece al path?
2. ¿Cuál pertenece al query y qué valor predeterminado tiene?
3. ¿Qué respuesta documenta FastAPI cuando la validación falla?

`/docs` no reemplaza las pruebas ni el diseño: hace visible el contrato generado a partir
de rutas y type hints.

Fuente: [documentación interactiva de FastAPI](https://fastapi.tiangolo.com/tutorial/first-steps/#interactive-api-docs).


### 11.1 Postman: prueba final de la API construida

Postman es la aplicación cliente que usaremos para enviar requests y observar responses.
Con el servidor de la API de viajes todavía activo, crea una solicitud `GET` a
`http://127.0.0.1:8000/viajes/42?pasajeros=2`, pulsa **Send** y registra su método, URL,
status, headers y body. Después prueba `/no-existe` y `/viajes/no-es-entero` para observar
`404` y `422`. Repite al menos una solicitud con `/docs` y `curl -i`; las tres herramientas
deben mostrar el mismo contrato HTTP.
La actividad forma parte de esta clase. Si la instalación falla, anota el mensaje y
continúa con la misma solicitud en `/docs` mientras resuelves el acceso a Postman.


## 12. Errores frecuentes y diagnóstico

| Síntoma | Qué revisar primero | Acción |
|---|---|---|
| `Address already in use` | Otro proceso usa el puerto 8000. | Detén el servidor anterior. Como alternativa usa `--port 8001` y cambia también la URL del cliente. |
| `Could not import module "main"` | Carpeta actual, nombre `main.py` y errores de sintaxis. | Ejecuta `pwd`, `ls` y `uv run python -m py_compile main.py`. |
| `Attribute "app" not found` | `main:app` exige un objeto llamado `app`. | Revisa `app = FastAPI(...)`. |
| El navegador no responde. | Servidor detenido o URL/puerto incorrectos. | Mira la terminal 1 antes de reiniciar cosas. |
| `422` con un número aparente. | Revisa qué parte de la URL recibió cada parámetro. | Lee el body JSON: contiene la ubicación y el tipo esperado. |
| Los cambios no aparecen. | Archivo sin guardar o `--reload` no está activo. | Guarda, observa el reinicio en logs y repite la solicitud. |

No ocultes el error: status, headers, body y logs son evidencia para localizar la capa que
falló.


## 🚀 Reto opcional

Agrega un query parameter opcional `detalle: bool = False` al endpoint de viajes y
devuélvelo en la respuesta. Antes de escribir, predice cómo llamarías al endpoint y qué
ocurrirá con `?detalle=true` y `?detalle=no-es-booleano`.

Comprueba el contrato en `/docs` y con `curl -i`. No agregues nuevos paquetes.


## Recopilación

Hoy construimos un mapa completo para conversar con una API:

- **API, cliente, servidor y recurso:** una API es la interfaz entre programas; el cliente
  pide un recurso y el servidor devuelve una respuesta.
- **Contrato:** acuerdo que define operaciones, entradas, respuestas y errores sin revelar
  la implementación interna.
- **REST:** estilo que organiza la API alrededor de recursos y métodos HTTP predecibles.
  SOAP, GraphQL y gRPC son alternativas con otras reglas.
- **Endpoint y nombramiento:** un endpoint combina método y ruta; usamos sustantivos,
  colecciones en plural y versionamiento como `/api/v1/viajes/`.
- **Request:** contiene línea inicial, URL, path, query, headers, línea vacía y body
  opcional. El path identifica, el query filtra o ajusta y el body transporta estructuras.
- **Response:** contiene código de estado, headers y body, normalmente JSON. Las familias
  `1xx`, `2xx`, `3xx`, `4xx` y `5xx` orientan la interpretación; hoy observamos `200`,
  `404` y `422`.
- **CRUD:** `GET` consulta, `POST` crea o procesa, `PUT`/`PATCH` actualiza y `DELETE`
  elimina. En esta clase implementamos sólo `GET`.
- **Seguridad:** autenticación verifica identidad, autorización verifica permisos y HTTPS
  protege el tránsito. Más adelante usaremos tokens.
- **Python y FastAPI:** los type hints comunican tipos; las clases pueden agrupar datos;
  un decorador modifica o registra una función. FastAPI usa esas ideas para declarar rutas,
  validar entradas y generar documentación.
- **Ejecución:** `uv run fastapi dev` inicia FastAPI; Uvicorn sirve la aplicación mediante
  el estándar ASGI. Postman, `/docs` y `curl` son clientes para observar el mismo contrato.

El ambiente reproducible de la clase 3 permite repetir esta aplicación. En la clase 5
ampliaremos el contrato con `POST`, modelos Pydantic y el diseño de las entradas del primer
modelo de predicción.


## ✅ Check final de la clase

Antes de detener el servidor con `Ctrl+C`, verifica:

- puedes distinguir interfaz, API, contrato, cliente, servidor, recurso y operación;
- puedes distinguir una URL completa, un path y un endpoint;
- puedes localizar línea inicial, headers, línea vacía y body en un request y un response;
- puedes justificar si un dato pertenece al path, query, header o body;
- puedes distinguir la intención de `GET`, `POST`, `PUT`/`PATCH` y `DELETE`;
- puedes distinguir HTTPS, autenticación, autorización y el manejo de secretos;
- `GET /` y `GET /viajes/{viaje_id}` responden `200` con JSON;
- puedes explicar por qué aparecen `404` y `422`;
- `/docs` y Postman coinciden con las rutas y type hints de `main.py`.

En la clase 5 ampliaremos el contrato para recibir datos con `POST`, cuerpos Pydantic y
errores controlados; usaremos ese contrato para diseñar las entradas del primer modelo de
predicción. La conversación HTTP de hoy será la base, no un ejemplo aislado.


## Tarea 2 — De funciones de viajes a una API local

La Tarea 2 transforma las funciones de estimación y resumen de viajes que desarrollaste
en las actividades iniciales en una API local con FastAPI y `uv`. Crearás los endpoints
`GET /api/v1/viajes` y `GET /api/v1/duracion/{distancia_km}` y verificarás su contrato
mediante clientes HTTP.

Consulta las [instrucciones completas de la Tarea 2](../docs/tareas/tarea-02-api-ambiente-uv.md)
antes de empezar. La entrega oficial se realiza en Canvas con la URL de tu pull request
fusionado y cerrado.


## 📚 Referencias

- [FastAPI: primeros pasos](https://fastapi.tiangolo.com/tutorial/first-steps/)
- [FastAPI: parámetros de path](https://fastapi.tiangolo.com/tutorial/path-params/)
- [FastAPI: parámetros de query](https://fastapi.tiangolo.com/tutorial/query-params/)
- [FastAPI: modelos Pydantic y tipos de Python](https://fastapi.tiangolo.com/python-types/#pydantic-models)
- [FastAPI: entornos e instalación con `uv`](https://fastapi.tiangolo.com/virtual-environments/)
- [FastAPI: servidores ASGI y Uvicorn](https://fastapi.tiangolo.com/deployment/manually/)
- [Especificación ASGI](https://asgi.readthedocs.io/en/latest/specs/main.html)
- [MDN: panorama general de HTTP](https://developer.mozilla.org/es/docs/Web/HTTP/Guides/Overview)
- [MDN: códigos de estado HTTP](https://developer.mozilla.org/es/docs/Web/HTTP/Reference/Status)
- [MDN: métodos HTTP](https://developer.mozilla.org/es/docs/Web/HTTP/Reference/Methods)
- [Documentación de Uvicorn](https://www.uvicorn.org/)
- [Postman: descarga](https://www.postman.com/downloads/) y [guía para enviar requests](https://learning.postman.com/docs/sending-requests/requests/)
- [Rick and Morty API](https://rickandmortyapi.com/)
- [Dad Jokes API](https://icanhazdadjoke.com/api#search-for-dad-jokes)

Las instrucciones escritas son suficientes para completar la práctica. Los recursos
enlazados sirven para consultar ejemplos y profundizar.
